<div style="background:linear-gradient(135deg, #1e3c72, #2a5298); border-radius:8px; padding:16px 20px; color:white;">
  <div style="font-size:80%; letter-spacing:2px; opacity:0.8;">LLM STUDIES · WEEK 2</div>
  <div style="font-size:150%; font-weight:700; margin-top:2px;">Remembering LLM Bases</div>
  <div style="font-size:90%; opacity:0.9; margin-top:6px;">Tokenizer → Embeddings → Positional Encoding</div>
</div>

**Dataset:** [Tiny Shakespeare](https://www.kaggle.com/datasets/kaushaltiwari/tiny-shakespeare) · **Runs on:** local dev container

<div style="background:#2a5298; border-radius:6px; padding:6px 14px; color:white; font-size:110%; font-weight:600;">
  <a id="index"></a>Contents
</div>

<div style="line-height:1.6; padding:2px 8px;">
<div style="margin-top:6px;"><a href="#sec-1"><b>1. Libraries</b></a></div>
<div style="margin-top:6px;"><a href="#sec-2"><b>2. Data</b></a></div>
<div style="margin-top:6px;"><a href="#sec-3"><b>3. Word-level tokenizer</b></a></div>
<div style="margin-left:22px;"><a href="#sec-3-1">3.1 Toy vocabulary</a></div>
<div style="margin-left:22px;"><a href="#sec-3-2">3.2 SimpleTokenizerV1</a></div>
<div style="margin-left:22px;"><a href="#sec-3-3">3.3 Validating the vocab</a></div>
<div style="margin-left:22px;"><a href="#sec-3-4">3.4 Unknown words: SimpleTokenizerV2 and &lt;|unk|&gt;</a></div>
<div style="margin-top:6px;"><a href="#sec-5"><b>5. Embeddings</b></a></div>
<div style="margin-left:22px;"><a href="#sec-5-1">5.1 Manual implementation</a></div>
<div style="margin-left:22px;"><a href="#sec-5-2">5.2 Linear implementation with torch</a></div>
<div style="margin-left:22px;"><a href="#sec-5-3">5.3 Data loader</a></div>
<div style="margin-left:22px;"><a href="#sec-5-4">5.4 Token embedding with torch</a></div>
<div style="margin-left:22px;"><a href="#sec-5-5">5.5 Positional embedding</a></div>
</div>

<div style="background:#2a5298; border-radius:6px; padding:6px 14px; color:white; font-size:110%; font-weight:600;">
  <a id="sec-1"></a>1. Libraries
</div>

In [12]:
import tiktoken
import torch

from src.config import DATA
from src.makemore import Makemore
from src.tokenizer import SimpleTokenizerV1, SimpleTokenizerV2, create_dataloader

torch.manual_seed(123)


<div style="background:#2a5298; border-radius:6px; padding:6px 14px; color:white; font-size:110%; font-weight:600;">
  <a id="sec-2"></a>2. Data
</div>

In [13]:
# Tokenizer using gt2
full_path = DATA / "tiny-shakespeare.txt"
with open(full_path, "r", encoding="utf-8") as f:
    text = f.read()

print(f"Length of dataset in characters: {len(text)}")

# makemore 
name_data = DATA / "names.txt"
names = open(name_data,  "r").read().splitlines()
print(f"Data for makemore: {names[:5]}")

Length of dataset in characters: 1115394
Data for makemore: ['emma', 'olivia', 'ava', 'isabella', 'sophia']


<div style="background:#2a5298; border-radius:6px; padding:6px 14px; color:white; font-size:110%; font-weight:600;">
  <a id="sec-3"></a>3. Word-level tokenizer
</div>

A tokenizer turns text into a list of integer ids (and back). Here we build the simplest one:
a word-level tokenizer backed by a dictionary `word -> id`.

<div style="border-left:3px solid #4f83e0; padding:1px 10px; font-weight:600;">
  <a id="sec-3-1"></a>3.1 Toy vocabulary
</div>

A vocabulary is just a dictionary. `sorted(set(...))` removes duplicates and makes the ids deterministic.

In [5]:
# To make it simple I will create a Toy Dictioary
toy_text = "I love my kids with all my heart"
splitted_text = toy_text.split(" ")
tokenizer = {word: idx for idx, word in enumerate(sorted(set(splitted_text)))}  # sorted -> deterministic ids
tokenizer["<unk>"] = len(tokenizer)
print(f"My tokenizer size `{len(tokenizer)}` {tokenizer}")

My tokenizer size `8` {'I': 0, 'all': 1, 'heart': 2, 'kids': 3, 'love': 4, 'my': 5, 'with': 6, '<unk>': 7}


<div style="border-left:3px solid #4f83e0; padding:1px 10px; font-weight:600;">
  <a id="sec-3-2"></a>3.2 SimpleTokenizerV1
</div>

`SimpleTokenizerV1` (in `src/tokenizer/base.py`) wraps the vocabulary:

- `encode`: splits the text with a regex (punctuation and spaces) and maps each piece to its id.
- `decode`: maps ids back to words and removes the space before punctuation.

The vocabulary includes two **special tokens** at the end: `<|endoftext|>` (separates documents)
and `<|unk|>` (replaces unknown words, used in 3.4). Adding them at the end keeps the other ids unchanged.

In [ ]:
# Vocabulary = words of the toy text + special tokens at the end
all_tokens = sorted(set(toy_text.split()))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token: idx for idx, token in enumerate(all_tokens)}
print(f"Vocab: {vocab}\n")

s_tokenizer = SimpleTokenizerV1(vocab)

# Testing
test_stnc = "my heart love kids"
test_ids = [1, 5, 2]
ids = s_tokenizer.encode(test_stnc.lower())

print(f"Encoding: {test_stnc} -> {ids}")
print(f"Decoding: {test_ids} -> {s_tokenizer.decode(test_ids)}")

<div style="border-left:3px solid #4f83e0; padding:1px 10px; font-weight:600;">
  <a id="sec-3-3"></a>3.3 Validating the vocab
</div>

How should the constructor check that `vocab` is valid? Options compared:

| Option | Verdict |
|---|---|
| `if not isinstance(...): raise TypeError` | ✅ Recommended for arguments coming from outside. Explicit, no dependencies. |
| `assert isinstance(...)` | ⚠️ Removed with `python -O` and raises `AssertionError` (means "bug in my code"). Use only for internal invariants and tests. |
| `typing.Annotated[...]` | ℹ️ Only metadata: Python does **not** check it at runtime. Useful with a library that reads it (e.g. pydantic `AfterValidator` + `@validate_call`). |
| Type hints + `mypy` (no runtime check) | ✅ Valid "EAFP" style for study code. |

Notes:

- Raise `TypeError` for a wrong type and `ValueError` for a right type with a wrong value.
- Accept `collections.abc.Mapping` instead of `dict` (duck typing).
- What really matters: **duplicate ids**. `{"a": 0, "b": 0}` silently loses an entry when building
  `int_to_str`, and no type checker catches it.

```python
@staticmethod
def _validate_vocab(vocab: object) -> dict[str, int]:
    if not isinstance(vocab, Mapping):
        raise TypeError(f"vocab must be a mapping, got {type(vocab).__name__}")
    if not all(isinstance(k, str) and isinstance(v, int) for k, v in vocab.items()):
        raise TypeError("vocab must map str -> int")
    if len(set(vocab.values())) != len(vocab):
        raise ValueError("vocab has duplicate token ids")
    return dict(vocab)  # own copy
```

<div style="border-left:3px solid #4f83e0; padding:1px 10px; font-weight:600;">
  <a id="sec-3-4"></a>3.4 Unknown words: SimpleTokenizerV2 and &lt;|unk|&gt;
</div>

`SimpleTokenizerV1` raises `KeyError` with any word that is not in the vocabulary.
`SimpleTokenizerV2` replaces unknown words with `<|unk|>` before the lookup.

**Pitfall:** `<|unk|>` itself must be in the vocabulary. Otherwise V2 replaces the word correctly
but then fails with `KeyError: '<|unk|>'`. That is why it was added to `vocab` in 3.2.
V2 could check it in its constructor: `if "<|unk|>" not in vocab: raise ValueError(...)`.

In [ ]:
# V1 fails with an unknown word
test_stnc = "I like to play video games"
try:
    ids = s_tokenizer.encode(test_stnc)
except KeyError as e:
    print(f"V1 tokenizer fail: unknown word -> {e.args[0]!r}")

# V2 maps unknown words to <|unk|> (needs <|unk|> in vocab)
s2_tokenizer = SimpleTokenizerV2(vocab)
ids = s2_tokenizer.encode(test_stnc)
print(f"V2 encoding: {test_stnc} -> {ids}")
print(f"V2 decoding: {ids} -> {s2_tokenizer.decode(ids)}")

<div style="background:#2a5298; border-radius:6px; padding:6px 14px; color:white; font-size:110%; font-weight:600;">
  <a id="sec-4"></a>4. Makemore
</div>

understading makemore which is an implementation to generate more name based on tainring data, this is an exercise to really practie the implementation of a tokenizer 

In [1]:
# Names Meatadata
print(f"Unique Names: {len(set(names))}")
print(f"Name Length: Min:{min(len(w) for w in set(names))}, Max:{max(len(w) for w in set(names))}")

# Create biagrams
more = Makemore(names)
# 
# #  
# biagrams = Makemore.bigram_counts(names)
# print(f"Unique combinations: {len(biagrams)}")
# print(f"Most repeated pattern: {biagrams[0][0]}, times:{biagrams[0][1]}")
# print(f"Less repeated pattern: {biagrams[-1][0]}, times:{biagrams[-1][1]}")

NameError: name 'names' is not defined

<div style="background:#2a5298; border-radius:6px; padding:6px 14px; color:white; font-size:110%; font-weight:600;">
  <a id="sec-5"></a>5. Embeddings
</div>

From token ids to vectors: the embedding matrix, the data loader and the positional embedding.

<div style="border-left:3px solid #4f83e0; padding:1px 10px; font-weight:600;">
  <a id="sec-5-1"></a>5.1 Manual implementation
</div>

Building (mimicking) a vector embedding by hand, using the toy `tokenizer` dictionary from 3.1.

An embedding maps each token id to a dense vector of real numbers.
It is stored as a matrix W of shape (N, M), where row i is the vector of token i:

- N = vocabulary size of the tokenizer (50257 for GPT-2 BPE), regardless of
how many of those tokens actually appear in our text.

- M = embedding dimension (e.g. 768 in GPT-2 small). It is a hyperparameter
we choose before training.

W starts with random values and is learned together with the rest of the model.
If we trained our own tokenizer on Tiny Shakespeare, N would be that tokenizer's
vocabulary size. A word-level vocabulary is impractical: any unseen word would
be out of vocabulary, which is why subword tokenizers like BPE are used.

In [ ]:
# Embdding Matrix
dim = 3
ids = torch.tensor([tokenizer[w] for w in splitted_text])
W = torch.arange(len(tokenizer) * dim, dtype=torch.float32).reshape(len(tokenizer), dim)
print(f"Embedding Matrix:\n{W}\n")

# Create One hot encoding
onehot = []
for idx in ids.tolist():
    row = [0] * len(tokenizer)
    row[idx] = 1
    onehot.append(row)

print(f"One hot encoding:\n{torch.tensor(onehot)}\n")

# Now create the embedding layer using the embedding matrix W
torch.tensor(onehot).float() @ W


Embedding Matrix:
tensor([[ 0.,  1.,  2.],
        [ 3.,  4.,  5.],
        [ 6.,  7.,  8.],
        [ 9., 10., 11.],
        [12., 13., 14.],
        [15., 16., 17.],
        [18., 19., 20.],
        [21., 22., 23.]])

One hot encoding:
tensor([[1, 0, 0, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 1, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0, 0]])



tensor([[ 0.,  1.,  2.],
        [ 3.,  4.,  5.],
        [12., 13., 14.],
        [ 9., 10., 11.],
        [18., 19., 20.],
        [15., 16., 17.],
        [12., 13., 14.],
        [ 6.,  7.,  8.]])

<div style="border-left:3px solid #4f83e0; padding:1px 10px; font-weight:600;">
  <a id="sec-5-2"></a>5.2 Linear implementation with torch
</div>

The same lookup, now as a `Linear` layer applied to one-hot vectors.

In [31]:
# Transform the ids into one-hot encoding using PyTorch's built-in function
onehot = torch.nn.functional.one_hot(ids, num_classes=len(tokenizer)).float()

# Now create the embedding layer using the embedding matrix W
linear = torch.nn.Linear(len(tokenizer), dim, bias=False)
linear.weight = torch.nn.Parameter(W.T)
linear(onehot)

tensor([[ 0.,  1.,  2.],
        [ 3.,  4.,  5.],
        [12., 13., 14.],
        [ 9., 10., 11.],
        [18., 19., 20.],
        [15., 16., 17.],
        [12., 13., 14.],
        [ 6.,  7.,  8.]], grad_fn=<MmBackward0>)

<div style="border-left:3px solid #4f83e0; padding:1px 10px; font-weight:600;">
  <a id="sec-5-3"></a>5.3 Data loader
</div>

This creates the dataset for training and testing. The challenge here is choosing `batch_size`, `max_length` and `stride`.

In [ ]:
# MaxLength is the maximum length of the sequence we want to feed into the model.
max_length = 128

# Generate Data
dataloader = create_dataloader(
    data=text,
    batch_size=64,
    max_length=max_length,
    stride=max_length,  # This avoid Overlapping Sequences
)

data_iter = iter(dataloader)
inputs, target = next(data_iter)
print("Inputs:\n", inputs)
print("target:\n", target)

Inputs:
 tensor([[   11, 15967,    11,  ...,   287,  5897,    11],
        [   77,  6243,  1275,  ...,   198,    40,   783],
        [  340,    11,   290,  ...,  1182,     0,   198],
        ...,
        [15876,    26,   290,  ...,   307,  1498,   284],
        [ 8001, 14210,  1654,  ...,    32,  1014,     0],
        [ 1719,   656,  3872,  ...,  3595,   582,    11]])
target:
 tensor([[15967,    11,   307,  ...,  5897,    11, 44642],
        [ 6243,  1275,   270,  ...,    40,   783,  2221],
        [   11,   290,   750,  ...,     0,   198,   198],
        ...,
        [   26,   290,   523,  ...,  1498,   284,  8791],
        [14210,  1654,   286,  ...,  1014,     0,  1309],
        [  656,  3872,    11,  ...,   582,    11,   616]])


<div style="border-left:3px solid #4f83e0; padding:1px 10px; font-weight:600;">
  <a id="sec-5-4"></a>5.4 Token embedding with torch
</div>

In [36]:
# Embeddings
output_dim = 256
vocab_size = tiktoken.get_encoding("gpt2").n_vocab  # 50257

# Embedding matrix W of shape (vocab_size, output_dim), randomly initialized.
# It is learned during training together with the rest of the model.
token_embedding_layer = torch.nn.Embedding(num_embeddings=vocab_size, embedding_dim=output_dim)

# Lookup: each token id in `inputs` selects its row of W
token_embedding = token_embedding_layer(inputs)
batch_size, seq_len, emb_dim = token_embedding.shape
print(f"Embedding output [batch_size, seq_len, emb_dim]: {token_embedding.shape}")
print(f"{batch_size} sequences, {seq_len} tokens each, {emb_dim} numbers per token")


Embedding output [batch_size, seq_len, emb_dim]: torch.Size([64, 128, 256])
64 sequences, 128 tokens each, 256 numbers per token


<div style="border-left:3px solid #4f83e0; padding:1px 10px; font-weight:600;">
  <a id="sec-5-5"></a>5.5 Positional embedding
</div>

From this point adding the positional encoding into the vector embedding to shift | move the vector for the transformer to identified the best vector position
we just sum them into the vector space , this would affect all the batches

In [ ]:
# Generate Positional Econding
post_embedding_layer = torch.nn.Embedding(num_embeddings=max_length, embedding_dim=output_dim)
print(f"Dim: {post_embedding_layer}", "\n")

input_embedding = token_embedding + post_embedding_layer(torch.arange(seq_len))
print(f"Input Embedding:\n{input_embedding}", "\n")

Dim: Embedding(128, 256) 

Input Embedding:
tensor([[[ 0.4694, -1.6320, -0.2709,  ...,  0.8581,  1.0578,  0.5506],
         [ 0.2702, -0.4971,  0.2619,  ...,  0.6610,  0.6027, -0.4364],
         [-2.1348, -1.9221,  0.5239,  ...,  0.1068,  0.8061, -0.9046],
         ...,
         [ 0.6609,  0.2691, -0.9726,  ..., -0.9408, -1.3474,  1.8791],
         [ 1.4877, -1.2918,  0.0621,  ..., -0.4917, -0.1761, -1.6591],
         [ 0.4364, -1.0930,  0.9721,  ..., -1.5924,  1.0923,  0.4269]],

        [[ 0.5229, -0.5372,  1.6073,  ...,  1.9726,  0.4796,  1.7309],
         [ 0.0774, -2.2603,  1.4476,  ..., -1.5105,  0.6517, -0.0900],
         [-2.4619, -0.6619, -1.7345,  ...,  1.2149, -0.0297, -1.9798],
         ...,
         [ 1.8562, -1.1432, -0.1695,  ..., -0.7692, -0.0972,  1.3120],
         [-0.2993, -1.8760, -2.2866,  ...,  0.5334,  2.3713, -0.1316],
         [-0.0591,  0.4070,  1.4472,  ..., -1.6189,  0.7885,  1.0540]],

        [[ 0.8773, -0.3089, -2.2383,  ...,  2.3252, -0.3874,  0.6993],
 